# P2 -- LLM-Powered Spatial Query Interface Demo

This notebook demonstrates the core components of the natural-language
spatial query pipeline:

1. Load a GeoPackage and display its schema
2. Show few-shot example queries
3. Demo SQL validation (good and bad queries)
4. Execute sample queries and display results

**Note:** LLM generation is mocked since no local LLM server is
available in this demo environment.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path
PROJECT_ROOT = Path.cwd().resolve().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared.data.generate_synthetic import generate_synthetic_geopackage

## 1. Generate / Load GeoPackage and Display Schema

In [ ]:
DATA_DIR = Path("../data/scratch")
DATA_DIR.mkdir(parents=True, exist_ok=True)

gpkg_path = generate_synthetic_geopackage(DATA_DIR)
print(f"GeoPackage created at: {gpkg_path}")

In [ ]:
from projects.p2_llm_spatial_query.src.schema_extractor import extract_schema

schema = extract_schema(gpkg_path)
for layer_name, layer_info in schema["layers"].items():
    print(f"\n=== {layer_name} ===")
    print(f"  Description: {layer_info['description']}")
    print("  Columns:")
    for col, info in layer_info["columns"].items():
        srid = info.get("srid", "")
        srid_str = f" (SRID:{srid})" if srid else ""
        print(f"    - {col}: {info['type']}{srid_str}")

## 2. Few-Shot Examples

In [ ]:
import yaml

few_shot_path = (
    PROJECT_ROOT / "projects" / "p2_llm_spatial_query"
    / "configs" / "few_shot_queries.yaml"
)
with open(few_shot_path) as f:
    few_shots = yaml.safe_load(f)

print(f"Loaded {len(few_shots['examples'])} few-shot examples:\n")
for i, ex in enumerate(few_shots["examples"][:5], 1):
    print(f"{i}. Q: {ex['question']}")
    print(f"   SQL: {ex['sql']}\n")

## 3. SQL Validation Demo

The validator is the critical safety component.  Let us test it with
both valid and malicious queries.

In [ ]:
from projects.p2_llm_spatial_query.src.sql_validator import validate_sql

safety_config = {
    "safety": {
        "allowed_operations": [
            "SELECT", "ST_Buffer", "ST_Intersects", "ST_Within",
            "ST_Contains", "ST_Area", "ST_Length",
        ],
        "blocked_operations": [
            "DELETE", "DROP", "UPDATE", "INSERT", "ALTER", "TRUNCATE",
        ],
    },
}

valid_spatial_sql = (
    "SELECT h.* FROM harvest_units h, streams s "
    "WHERE ST_Intersects(h.geometry, ST_Buffer(s.geometry, 60.96))"
)

test_cases = [
    ("Valid SELECT", "SELECT * FROM harvest_units WHERE acres > 40"),
    ("Valid spatial", valid_spatial_sql),
    ("BLOCKED: DELETE", "DELETE FROM harvest_units WHERE unit_id = 1"),
    ("BLOCKED: DROP", "DROP TABLE harvest_units"),
    ("BLOCKED: injection", "SELECT * FROM harvest_units; DROP TABLE streams"),
    ("BLOCKED: bad func", "SELECT ST_Union(geometry) FROM harvest_units"),
]

for label, sql in test_cases:
    is_valid, reason = validate_sql(sql, safety_config)
    status = "PASS" if is_valid else "BLOCKED"
    print(f"[{status}] {label}")
    if reason:
        print(f"         Reason: {reason}")

## 4. Execute Sample Queries

In [ ]:
from projects.p2_llm_spatial_query.src.executor import execute_query
from projects.p2_llm_spatial_query.src.formatter import format_results

config = {"geopackage": {"path": str(gpkg_path)}}

# Query 1: All harvest units
sql = "SELECT unit_id, unit_name, acres, prescription FROM harvest_units"
result = execute_query(sql, gpkg_path, config)
print(format_results(result, "Show all harvest units"))

In [ ]:
# Query 2: Aggregate -- total clearcut acreage
sql = "SELECT SUM(acres) AS total_clearcut_acres FROM harvest_units WHERE prescription = 'clearcut'"
result = execute_query(sql, gpkg_path, config)
print(format_results(result, "Total acreage of clearcut units"))

In [ ]:
# Query 3: Stream classes
sql = "SELECT stream_id, stream_class, name FROM streams"
result = execute_query(sql, gpkg_path, config)
print(format_results(result, "List all streams"))

In [ ]:
# Query 4: Sensitive habitats
sql = "SELECT habitat_id, species, status FROM sensitive_habitats"
result = execute_query(sql, gpkg_path, config)
print(format_results(result, "Show sensitive habitat areas"))

## 5. Prompt Construction Demo

Demonstrates how the system and user prompts are assembled.

In [ ]:
from projects.p2_llm_spatial_query.src.prompt_builder import (
    build_system_prompt,
    build_user_prompt,
    select_few_shots,
)

schema_meta_path = (
    PROJECT_ROOT / "projects" / "p2_llm_spatial_query"
    / "configs" / "schema_metadata.yaml"
)
with open(schema_meta_path) as f:
    schema_meta = yaml.safe_load(f)

system_prompt = build_system_prompt(schema_meta, safety_config)
print("=== System Prompt (first 800 chars) ===")
print(system_prompt[:800])
print("...")

In [ ]:
user_query = "Find harvest units within 200 feet of Class I streams"
selected = select_few_shots(user_query, few_shots["examples"], top_k=3)

user_prompt = build_user_prompt(user_query, selected, safety_config)
print("=== User Prompt ===")
print(user_prompt)

## Summary

This notebook demonstrated the offline-capable components of the P2 pipeline:
schema extraction, few-shot selection, prompt construction, SQL validation,
and query execution.  In production, the `pipeline.run_query()` function
connects these stages with a local LLM endpoint (e.g. Llama 3 via vLLM)
to translate free-form natural language into validated spatial SQL.